# Embedding the VaR backtest report in a notebook

`report.html` is script-driven, so it won't render in a Markdown cell &mdash;
Jupyter strips `<script>` (and `display(HTML(...))` is sanitized the same way in
JupyterLab). The reliable approach is an **iframe**, which runs its own JS and
shows the report right in the output cell.

Run the code cell below. `show_report()`:
1. searches the current working directory (and its subfolders) for `report.html`
   and `var_backtest_results.csv`;
2. if the CSV is missing &mdash; it's gitignored, so it won't be in a fresh clone
   &mdash; it regenerates it by running `generate_data.py` then `var_backtest.py`
   (needs `pandas`, `numpy`, `scipy`, `openpyxl` in the kernel's environment);
3. bakes the CSV into the page, writes `report_embedded.html` in the working
   directory, and displays it inline via `IFrame`.

**`report.html` vs `report_embedded.html`**
- `report.html` is the reusable report app from the repo: it has a file-picker
  and no data inside, so on its own you'd open it and choose a CSV.
- `report_embedded.html` is what this cell writes each run: a copy of
  `report.html` with *your* CSV baked in, so it shows your data immediately with
  no picker. It's disposable (regenerated every run, gitignored) and is what the
  iframe displays.

**Tips**
- If auto-discovery finds the wrong file, pass explicit paths (forward slashes
  work on Windows too):
  `show_report(html_path="C:/path/to/report.html", csv_path="C:/path/to/var_backtest_results.csv")`
- The printed links use proper `file://` URIs, so paths with spaces (e.g.
  `OneDrive - Dimensional`) stay intact. `report_embedded.html` is fully
  self-contained, so you can also just open it directly in a browser tab.

In [ ]:
# Display the standalone VaR report inline. Auto-finds the repo files under the
# current working directory, so it works even when the notebook lives elsewhere.
import base64, sys, subprocess
from pathlib import Path
from IPython.display import IFrame, HTML, display

def _find(name, root, explicit=None):
    if explicit:
        p = Path(explicit).expanduser()
        if not p.exists():
            raise FileNotFoundError(f"{p} not found.")
        return p
    root = Path(root)
    if (root / name).exists():
        return root / name
    hits = sorted(root.rglob(name))   # search subfolders (e.g. a cloned repo dir)
    return hits[0] if hits else None

def show_report(html_path=None, csv_path=None, search_root=None,
                height=900, generate_if_missing=True):
    root = Path(search_root).expanduser() if search_root else Path.cwd()

    html = _find("report.html", root, html_path)
    if html is None:
        raise FileNotFoundError(
            f"report.html not found under {root}.\n"
            "Pull the backtest-exceedances repo there, or pass html_path=...")
    repo = html.parent

    # The CSV is gitignored, so it may be absent in a fresh clone.
    csv = Path(csv_path) if csv_path else (repo / "var_backtest_results.csv")
    if not csv.exists():
        found = _find("var_backtest_results.csv", root)
        if found is not None:
            csv = found
    if not csv.exists() and generate_if_missing:
        gen, bt = repo / "generate_data.py", repo / "var_backtest.py"
        if gen.exists() and bt.exists():
            print("var_backtest_results.csv missing - generating it from the scripts...")
            try:
                subprocess.run([sys.executable, str(gen)], cwd=repo, check=True)
                subprocess.run([sys.executable, str(bt)], cwd=repo, check=True)
                csv = repo / "var_backtest_results.csv"
            except Exception as e:
                raise RuntimeError(
                    f"Auto-generate failed ({e}). Run generate_data.py then var_backtest.py "
                    f"manually in {repo} (needs pandas, numpy, scipy, openpyxl).")
    if not csv.exists():
        raise FileNotFoundError(
            "var_backtest_results.csv not found (gitignored, so not in a fresh clone).\n"
            f"Run generate_data.py then var_backtest.py in {repo}, or pass csv_path=...")

    # Bake the data in and write the embedded file next to the NOTEBOOK (cwd), so
    # the Jupyter server serves it via a simple relative iframe src.
    page = html.read_text(encoding="utf-8")
    b64 = base64.b64encode(csv.read_bytes()).decode()
    page = page.replace(
        "</body>",
        f'<script>window.addEventListener("load",()=>loadText(atob("{b64}")));</script></body>')
    out = Path.cwd() / "report_embedded.html"
    out.write_text(page, encoding="utf-8")

    # Clickable links via as_uri() so spaces in the path (e.g. "OneDrive - Dimensional")
    # are percent-encoded and don't get cut off.
    def link(p, note=""):
        p = p.resolve()
        tail = f' <span style="color:#6b7280">{note}</span>' if note else ""
        return f'<a href="{p.as_uri()}" target="_blank" title="{p}">{p.name}</a>{tail}'
    display(HTML(
        '<div style="font:12px/1.6 -apple-system,Segoe UI,sans-serif">'
        f'app template: {link(html, "(file-picker, no data baked in)")}<br>'
        f'data source: {link(csv)}<br>'
        f'standalone view: {link(out, "(your data baked in - opens full-page)")}'
        '</div>'))
    return IFrame(out.name, width="100%", height=height)

show_report()